# Notebook to process ParseBio data - PB-002

In [ ]:
pip list #  (scvi-r docker kernel is used docker pull alexanrna/scanpy-r:1.10-v6 )

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
import numpy as np 
from scipy import stats
import glob 
import os
import re
from matplotlib import pyplot as plt
import datetime
import anndata as ad
from scipy import sparse
from anndata import AnnData
from scipy.sparse import csr_matrix, issparse
from collections import OrderedDict

In [ ]:
sc.settings.verbosity = 3             # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.logging.print_header()

sc.settings.set_figure_params(
    dpi=80,
    facecolor="white",
    frameon=False,
)


In [ ]:
mat_path = '/PB-002/PB-002-comb/all-sample/DGE_filtered/' # change path to parsebio PB-002 DGE_filtered folder as necessary

In [ ]:
# The DGE_filtered folder contains the expression matrix, genes, and files
adata = sc.read_mtx(mat_path + 'count_matrix.mtx')


# reading in gene and cell data
gene_data = pd.read_csv(mat_path + 'all_genes.csv')
cell_meta = pd.read_csv(mat_path + 'cell_metadata.csv')

# find genes with nan values and filter
gene_data = gene_data[gene_data.gene_name.notnull()]
notNa = gene_data.index
notNa = notNa.to_list()

# remove genes with nan values and assign gene names
adata = adata[:,notNa]
adata.var = gene_data
adata.var.set_index('gene_name', inplace=True)
adata.var.index.name = None
adata.var_names_make_unique()

# add cell meta data to anndata object
adata.obs = cell_meta
adata.obs.set_index('bc_wells', inplace=True)
adata.obs.index.name = None
adata.obs_names_make_unique()

sc.pp.filter_cells(adata, min_counts=100)
sc.pp.filter_genes(adata, min_cells=5)
adata.shape

In [ ]:
adata

In [ ]:
adata.layers["counts"] = adata.X.copy()

In [ ]:
sc.pp.filter_cells(adata, min_counts=200)

In [ ]:
adata

# Quick QC visualisations

In [ ]:
sc.pl.highest_expr_genes(adata, n_top=20, )

In [ ]:
# mitochondrial genes
adata.var["mt"] = adata.var_names.str.startswith("MT-")
# ribosomal genes
adata.var["ribo"] = adata.var_names.str.startswith(("RPS", "RPL"))
# hemoglobin genes.
adata.var["hb"] = adata.var_names.str.contains(("^HB[^(P)]"))

In [ ]:
sc.pp.calculate_qc_metrics(
    adata, qc_vars=["mt", "ribo", "hb"], inplace=True, percent_top=[20], log1p=True
)
adata

In [ ]:
p1 = sns.displot(adata.obs["total_counts"], bins=100, kde=False)
# sc.pl.violin(adata, 'total_counts')
p2 = sc.pl.violin(adata, "pct_counts_mt")
p3 = sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")
p4 = sc.pl.violin(adata, "pct_counts_ribo")
p4 = sc.pl.violin(adata, "pct_counts_hb")

In [ ]:
metric = "pct_counts_mt"

M = adata.obs[metric]
fig, ax = plt.subplots()
ax.hist(adata.obs[metric], bins=100)
ax.set_xlabel(metric)
ax.set_ylabel('Frequency')
ax.legend(['Data'])
plt.show()

In [ ]:
p1 = sns.histplot(adata.obs["total_counts"], bins=100, kde=False)

## Detect MAD outliers

In [ ]:
def is_outlier(adata, metric: str, nmads: int):
    """
    calculate median absolute deviation (MAD) to get find outliers
    """
    M = adata.obs[metric]
    outlier = (M < np.median(M) - nmads * stats.median_abs_deviation(M)) | (
        np.median(M) + nmads * stats.median_abs_deviation(M) < M
    )
    return outlier

In [ ]:
adata.obs["outlier"] = (
        is_outlier(adata, "log1p_total_counts", 5)
        | is_outlier(adata, "log1p_n_genes_by_counts", 5)
        | is_outlier(adata, "pct_counts_in_top_20_genes", 5)
        )
print(adata.obs.outlier.value_counts())

In [ ]:
adata = adata[(~adata.obs.outlier)].copy()

In [ ]:
# mitochondria
adata.obs["mt_outlier"] = is_outlier(adata, "pct_counts_mt", 5) | (
    adata.obs["pct_counts_mt"] > 12.5
    )
print(adata.obs.mt_outlier.value_counts())

In [ ]:
adata = adata[(~adata.obs.mt_outlier)].copy()

# Normalisation

In [ ]:
# Normalizing to median total counts
sc.pp.normalize_total(adata)
# Logarithmize the data:
sc.pp.log1p(adata)
adata.layers["log1p_norm"] = adata.X.copy()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
p1 = sns.histplot(adata.obs["total_counts"], bins=100, kde=False, ax=axes[0])
axes[0].set_title("Total counts")
p2 = sns.histplot(adata.layers["log1p_norm"].sum(1), bins=100, kde=False, ax=axes[1])
axes[1].set_title("Shifted logarithm")
plt.show()

# Feature selection

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes=3000)

In [ ]:
sc.pl.highly_variable_genes(adata)

# Dimensionality reduction

In [ ]:
sc.pp.pca(adata)
sc.pl.pca_variance_ratio(adata, n_pcs=50, log=True)
sc.pl.pca_scatter(adata, color="total_counts")

In [ ]:
sc.pp.neighbors(adata)
sc.tl.umap(adata)

In [ ]:
sc.tl.tsne(adata)

In [ ]:
res = 1.2
sc.tl.leiden(adata, resolution=res, key_added="leiden_" + str(res))
res = 1.5
sc.tl.leiden(adata, resolution=res, key_added="leiden_" + str(res))

In [ ]:
sc.pl.umap(
    adata,
    color=["total_counts", "pct_counts_mt"] 
)

In [ ]:
sc.pl.tsne(
    adata,
    color=["total_counts", "pct_counts_mt"]
)

In [ ]:
adata

In [ ]:
# date = datetime.datetime.now().strftime('%Y%m%d')
# adata.X = adata.layers['counts'].copy()
# adata.write(f'./matrices/{date}_PB-002_rna_filtered_umap_tsne.h5ad')

In [ ]:
sc.pl.umap(adata, color = "MALAT1")

# Metadata

## Region annotation 

In [ ]:
# create a dictionary to map each multiome run to its brain region of origin
cluster2annotation = {
"pool_1":"Substantia Nigra",
    "pool_2":"Substantia Nigra",
    "pool_3":"Cingulate Gyrus",
    "pool_4":"Cingulate Gyrus",
    "pool_5":"Cingulate Gyrus",
    "pool_6":"Cingulate Gyrus",
    "pool_7":"Cingulate Gyrus",
    "pool_8":"Cingulate Gyrus",
    "pool_9":"Substantia Nigra",
    "pool_10":"Substantia Nigra"
    
}

adata.obs["region"] = (
    adata.obs["sample"].map(cluster2annotation).astype("category")
)

In [ ]:
adata.obs

# Vireo assignment <br>
## Donor assignments from Vireo/CellSNP

In [ ]:
vireo_files = glob.glob('vireo_PB-002/*')

In [ ]:
vireo_files

In [ ]:
def append_name_if_donor(row):
    if re.search(r'donor', row['donor_id'], re.IGNORECASE):
        return name + '_' + row['donor_id']
    return row['donor_id']

In [ ]:
df_merged = pd.DataFrame()
for file in vireo_files:
    print(file)
    name=file.replace('vireo/','')
    name=name.split('.')[0]
    print('\t' + name)
    
    df=pd.read_csv(file, sep='\t')

    df['donor_id'] = df.apply(append_name_if_donor, axis=1)
    df['full_barcode'] = df['cell'] 
    df['full_barcode']
    df.index = df['full_barcode']
    df = df[df.index.isin(adata.obs.index)]
    print(f'\t{str(len(df))}')
    df_merged = pd.concat([df_merged, df])

In [ ]:
df_merged['donor_id'].head()

In [ ]:
df_merged['donor_id'].value_counts()

In [ ]:
def print_full(x):
    pd.set_option('display.max_rows', len(x))
    print(x)
    pd.reset_option('display.max_rows')

In [ ]:
print_full(df_merged['donor_id'].value_counts())

In [ ]:
adata.obs['vireo_assignment'] = df_merged['donor_id']

In [ ]:
adata.obs['vireo_assignment'].head()

In [ ]:
adata.obs['vireo_assignment'].value_counts()

In [ ]:
sc.pl.umap(adata, color = "vireo_assignment")

In [ ]:
sc.pl.umap(adata, color = "vireo_assignment", groups = 'unassigned')

In [ ]:
sc.pl.umap(adata, color = "vireo_assignment", groups = 'doublet')

In [ ]:
adata

In [ ]:
print_full(adata.obs.groupby(by=["vireo_assignment","region"]).size())

In [ ]:
# replace missing genome donor 
adata.obs['vireo_assignment'] = adata.obs['vireo_assignment'].replace('vireo_PB-002/PB-002-pool_5_donor9', 'ASA_137')
adata.obs['vireo_assignment'][adata.obs['sample'] == 'pool_5'].value_counts()[:15]

# Check some marker genes

In [ ]:
marker_genes = {
    "neurons": ["SYT1"],
    "OLG": ["MAG", "MOBP"],
    "OPC": [
        "PDGFRA"],
    "Ast":["AQP4", "GFAP"],
    "Micro":["CD74","RUNX1"],
    "Endo":["CLDN5"]}

In [ ]:
SN = {
    "neurons",
    "OLG",
    "OPC",
    "Ast",
    "Micro",
    "Endo"}

In [ ]:
marker_genes_in_data = dict()
for ct, markers in marker_genes.items():
    markers_found = list()
    for marker in markers:
        if marker in adata.var.index:
            markers_found.append(marker)
    marker_genes_in_data[ct] = markers_found

In [ ]:
for ct in SN:
    print(f"{ct.upper()}:")  # print cell subtype name
    sc.pl.tsne(
        adata,
        color=marker_genes_in_data[ct],
        vmin=0,
        vmax="p99", 
        sort_order=False,  
        frameon=False,
        cmap="Reds",  
    )
    print("\n\n\n")  

# Save data

In [ ]:
date = datetime.datetime.now().strftime('%Y%m%d')
adata.X = adata.layers["counts"].copy()

In [ ]:
# subset data
adata_sn = adata[(adata.obs["region"].isin(["Substantia Nigra"]))]
adata_sn

In [ ]:
adata_cortex = adata[(adata.obs["region"].isin(["Cingulate Gyrus"]))]
adata_cortex

In [ ]:
# save  the data
date = datetime.datetime.now().strftime('%Y%m%d')
# substantia nigra
adata_sn.write(f'./matrices/{date}_all_PB-002_sn.h5ad')
# cingulate gyrus and motor cortex
adata_cortex.write(f'./matrices/{date}_all_PB-002_cg.h5ad')